# 03 - HMM Regime Detection

This notebook trains a Gaussian Hidden Markov Model (HMM) to discover market regimes such as bull, bear, and crisis from the engineered market features.

Input: `../data/processed/features.csv`  
Output: `../data/processed/regime_features.csv`

In [1]:
%pip install hmmlearn scikit-learn

  Using cached hmmlearn-0.3.3-cp311-cp311-win_amd64.whl.metadata (3.1 kB)
  Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl.metadata (11 kB)
  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached hmmlearn-0.3.3-cp311-cp311-win_amd64.whl (127 kB)
Using cached scikit_learn-1.8.0-cp311-cp311-win_amd64.whl (8.1 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl (12.6 MB)
Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl (36.6 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%pip install pandas

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
try:
	import matplotlib.pyplot as plt
except ModuleNotFoundError:
	get_ipython().run_line_magic("pip", "install matplotlib seaborn")
	import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler

# Import GaussianHMM; if the import fails at runtime, install and retry.
try:
	from hmmlearn.hmm import GaussianHMM
except ModuleNotFoundError:
	# Ensure package is installed in the current kernel and retry import
	get_ipython().run_line_magic("pip", "install hmmlearn scikit-learn")
	from hmmlearn.hmm import GaussianHMM

plt.style.use("seaborn-v0_8-whitegrid")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'matplotlib'

## 2. Load Feature Dataset

In [ ]:
FEATURE_PATH = Path("../data/processed/features.csv")
OUTPUT_PATH = Path("../data/processed/regime_features.csv")

features = pd.read_csv(FEATURE_PATH, parse_dates=["Date"])
features = features.sort_values(["ticker", "Date"]).reset_index(drop=True)

print(features.shape)
print(features["ticker"].value_counts())
features.head()

## 3. Select Asset And HMM Features

Start with NIFTY 50 (`^NSEI`). Later, you can repeat the same process for S&P 500 and NASDAQ.

In [ ]:
PRIMARY_TICKER = "^NSEI"

if PRIMARY_TICKER not in features["ticker"].unique():
    PRIMARY_TICKER = features["ticker"].unique()[0]
    print(f"^NSEI not found. Using {PRIMARY_TICKER} instead.")

hmm_feature_cols = [
    "return_1d",
    "volatility_30d",
    "momentum_30d",
    "ma_50_ratio",
    "ma_200_ratio",
    "drawdown",
    "vix_close",
]

asset_df = features[features["ticker"] == PRIMARY_TICKER].copy()
asset_df = asset_df.dropna(subset=hmm_feature_cols).reset_index(drop=True)

print("Selected ticker:", PRIMARY_TICKER)
print("Rows available:", len(asset_df))
asset_df[["Date", "ticker", "Close"] + hmm_feature_cols].head()

## 4. Scale Features

HMMs are sensitive to feature scale, so we standardize all model inputs.

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(asset_df[hmm_feature_cols])

print("Training matrix shape:", X.shape)

## 5. Train Gaussian HMM

We use 3 hidden states as a first version:

- Bull regime
- Bear regime
- Crisis / high-volatility regime

In [ ]:
N_REGIMES = 3

hmm_model = GaussianHMM(
    n_components=N_REGIMES,
    covariance_type="full",
    n_iter=1000,
    random_state=42,
)

hmm_model.fit(X)

asset_df["regime"] = hmm_model.predict(X)
regime_probabilities = hmm_model.predict_proba(X)

for i in range(N_REGIMES):
    asset_df[f"regime_{i}_prob"] = regime_probabilities[:, i]

asset_df[["Date", "ticker", "Close", "regime", "regime_0_prob", "regime_1_prob", "regime_2_prob"]].head()

## 6. Understand And Label Regimes

HMM state numbers are arbitrary. This cell summarizes each state so we can map them to human labels.

In [ ]:
regime_summary = (
    asset_df
    .groupby("regime")
    .agg(
        observations=("regime", "size"),
        avg_daily_return=("return_1d", "mean"),
        annualized_return=("return_1d", lambda x: x.mean() * 252),
        annualized_volatility=("return_1d", lambda x: x.std() * np.sqrt(252)),
        avg_momentum_30d=("momentum_30d", "mean"),
        avg_drawdown=("drawdown", "mean"),
        worst_drawdown=("drawdown", "min"),
        avg_vix=("vix_close", "mean"),
    )
    .sort_index()
)

regime_summary

In [ ]:
def label_regimes(summary):
    """
    Simple rule-based labels:
    - Crisis: highest volatility, unless another state has much worse drawdown and negative return.
    - Bull: best annualized return among remaining states.
    - Bear: remaining state.
    """
    summary = summary.copy()

    crisis_state = summary["annualized_volatility"].idxmax()

    remaining = [state for state in summary.index if state != crisis_state]
    bull_state = summary.loc[remaining, "annualized_return"].idxmax()

    bear_state = [state for state in summary.index if state not in [crisis_state, bull_state]][0]

    return {
        bull_state: "Bull",
        bear_state: "Bear",
        crisis_state: "Crisis",
    }

regime_label_map = label_regimes(regime_summary)
asset_df["regime_label"] = asset_df["regime"].map(regime_label_map)

print(regime_label_map)
asset_df[["Date", "Close", "regime", "regime_label"]].head()

## 7. Visualize Regimes On Price Chart

In [ ]:
colors = {
    "Bull": "green",
    "Bear": "orange",
    "Crisis": "red",
}

plt.figure(figsize=(14, 6))
plt.plot(asset_df["Date"], asset_df["Close"], color="black", linewidth=1.2, label="Close")

for label, color in colors.items():
    temp = asset_df[asset_df["regime_label"] == label]
    plt.scatter(temp["Date"], temp["Close"], s=8, color=color, label=label, alpha=0.7)

plt.title(f"{PRIMARY_TICKER} HMM Market Regimes")
plt.xlabel("Date")
plt.ylabel("Close")
plt.legend()
plt.show()

## 8. Regime Timeline

In [ ]:
regime_numeric = asset_df["regime_label"].map({"Bull": 2, "Bear": 1, "Crisis": 0})

plt.figure(figsize=(14, 3))
plt.plot(asset_df["Date"], regime_numeric, drawstyle="steps-post")
plt.yticks([0, 1, 2], ["Crisis", "Bear", "Bull"])
plt.title(f"{PRIMARY_TICKER} Regime Timeline")
plt.xlabel("Date")
plt.ylabel("Regime")
plt.show()

## 9. Save Regime Dataset

This output becomes useful for the next notebook: Mixture of Experts.

In [ ]:
regime_output_cols = [
    "Date",
    "ticker",
    "Close",
    "return_1d",
    "volatility_30d",
    "momentum_30d",
    "ma_50_ratio",
    "ma_200_ratio",
    "drawdown",
    "vix_close",
    "target_return_1d",
    "target_direction_1d",
    "regime",
    "regime_label",
    "regime_0_prob",
    "regime_1_prob",
    "regime_2_prob",
]

regime_dataset = asset_df[regime_output_cols].copy()
regime_dataset.to_csv(OUTPUT_PATH, index=False)

print("Saved regime dataset to:", OUTPUT_PATH)
print(regime_dataset.shape)
regime_dataset.head()

## 10. Optional: Train Regimes For Every Asset

This creates HMM regimes separately for every ticker in your feature dataset. Use this when you want a broader dataset for regime-aware modeling.

In [ ]:
def fit_hmm_for_ticker(df, ticker, feature_cols, n_regimes=3):
    temp = df[df["ticker"] == ticker].copy()
    temp = temp.dropna(subset=feature_cols).reset_index(drop=True)

    if len(temp) < 300:
        print(f"Skipping {ticker}: not enough rows")
        return None

    X_temp = StandardScaler().fit_transform(temp[feature_cols])

    model = GaussianHMM(
        n_components=n_regimes,
        covariance_type="full",
        n_iter=1000,
        random_state=42,
    )
    model.fit(X_temp)

    temp["regime"] = model.predict(X_temp)
    probs = model.predict_proba(X_temp)

    for i in range(n_regimes):
        temp[f"regime_{i}_prob"] = probs[:, i]

    summary = (
        temp
        .groupby("regime")
        .agg(
            observations=("regime", "size"),
            annualized_return=("return_1d", lambda x: x.mean() * 252),
            annualized_volatility=("return_1d", lambda x: x.std() * np.sqrt(252)),
            avg_drawdown=("drawdown", "mean"),
            avg_vix=("vix_close", "mean"),
        )
    )

    labels = label_regimes(summary)
    temp["regime_label"] = temp["regime"].map(labels)

    return temp

all_regime_dfs = []

for ticker in features["ticker"].unique():
    result = fit_hmm_for_ticker(features, ticker, hmm_feature_cols)
    if result is not None:
        all_regime_dfs.append(result)

all_regimes = pd.concat(all_regime_dfs, ignore_index=True)
all_regimes.to_csv("../data/processed/regime_features_all_assets.csv", index=False)

print(all_regimes.shape)
print(all_regimes["ticker"].value_counts())
all_regimes.head()

## Next Step

After this notebook runs successfully, build `04_mixture_of_experts.ipynb`.

That notebook will compare:

- Baseline single model
- Regime-aware model with separate experts